In [1]:
import os, psutil, torch
from google.colab import drive

ram_gb = psutil.virtual_memory().total / (1024 ** 3)
print(f"System RAM : {ram_gb:.1f} GB")
print(f"CUDA ready : {torch.cuda.is_available()}")
assert torch.cuda.is_available(), "Chưa bật GPU! Hãy chọn L4 GPU trong Change runtime type."
print(f"GPU name   : {torch.cuda.get_device_name(0)}")
!nvidia-smi

drive.mount('/content/drive')

System RAM : 53.0 GB
CUDA ready : True
GPU name   : NVIDIA L4
Wed Nov 25 09:30:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   40C    P8             13W /   72W |       3MiB /  23034MiB |      0%      Default |
|                                         |                        |          

In [4]:
%%bash
set -e
cd /content
if [ ! -d "frn-50k-baseline" ]; then
    git clone https://github.com/Dingdong-Inc/frn-50k-baseline.git
fi
cd /content/frn-50k-baseline
git checkout 8af694a09a5a427d1d50648d348c022565a781f2
git log -n 1 --oneline

echo "=== Cây thư mục demand_forecasting/TFT ==="
find /content/frn-50k-baseline/demand_forecasting/TFT -maxdepth 3 -not -path '*/.*'

8af694a fix a bug due to .gitignore
=== Cây thư mục demand_forecasting/TFT ===
/content/frn-50k-baseline/demand_forecasting/TFT
/content/frn-50k-baseline/demand_forecasting/TFT/configs
/content/frn-50k-baseline/demand_forecasting/TFT/configs/tft_config.py
/content/frn-50k-baseline/demand_forecasting/TFT/configs/__init__.py
/content/frn-50k-baseline/demand_forecasting/TFT/data
/content/frn-50k-baseline/demand_forecasting/TFT/data/timeseries.py
/content/frn-50k-baseline/demand_forecasting/TFT/data/__init__.py
/content/frn-50k-baseline/demand_forecasting/TFT/trainer
/content/frn-50k-baseline/demand_forecasting/TFT/trainer/model.py
/content/frn-50k-baseline/demand_forecasting/TFT/trainer/__init__.py
/content/frn-50k-baseline/demand_forecasting/TFT/predictTFT.py
/content/frn-50k-baseline/demand_forecasting/TFT/models
/content/frn-50k-baseline/demand_forecasting/TFT/models/base_model.py
/content/frn-50k-baseline/demand_forecasting/TFT/models/tft
/content/frn-50k-baseline/demand_forecasting/T

Cloning into 'frn-50k-baseline'...
Note: switching to '8af694a09a5a427d1d50648d348c022565a781f2'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is now at 8af694a fix a bug due to .gitignore


In [8]:
%%bash
set -e
export MPLBACKEND=Agg
export MAMBA_ROOT_PREFIX=/content/micromamba
PY_BIN="$MAMBA_ROOT_PREFIX/envs/tft_py310/bin/python"

# Kiểm định toàn bộ phiên bản, GPU L4 và import thẳng các module TFT của repo
$PY_BIN -c "
import sys, torch, numpy, pandas, pyarrow, datasets, pytorch_lightning, lightning, pytorch_forecasting
print('Python              :', sys.version.split()[0])
print('torch               :', torch.__version__, '| CUDA:', torch.cuda.is_available())
print('GPU                 :', torch.cuda.get_device_name(0))
print('pytorch_lightning   :', pytorch_lightning.__version__)
print('lightning           :', lightning.__version__)
print('pytorch_forecasting :', pytorch_forecasting.__version__)
print('numpy               :', numpy.__version__)
print('pandas              :', pandas.__version__)
print('pyarrow             :', pyarrow.__version__)
assert sys.version_info[:2] == (3, 10)
assert torch.__version__.startswith('2.0.1')
assert pytorch_lightning.__version__ == '1.9.4'
assert pytorch_forecasting.__version__ == '1.0.0'
assert int(numpy.__version__.split('.')[0]) < 2
assert tuple(map(int, pandas.__version__.split('.')[:2])) < (2, 2)
assert torch.cuda.is_available()

x = torch.randn(64, 64, device='cuda') @ torch.randn(64, 64, device='cuda')
print('L4 CUDA matmul check: OK | shape =', tuple(x.shape))

sys.path.insert(0, '/content/frn-50k-baseline/demand_forecasting/TFT')
import configs.tft_config as config
from trainer.model import Model
from dataset.dataset import Dataset
from models.tft.model import TemporalFusionTransformer
print('Repo TFT modules import: OK!')
"

Python              : 3.10.22
torch               : 2.0.1+cu118 | CUDA: True
GPU                 : NVIDIA L4
pytorch_lightning   : 1.9.4
lightning           : 2.0.9
pytorch_forecasting : 1.0.0
numpy               : 1.26.4
pandas              : 2.1.4
pyarrow             : 25.0.1
L4 CUDA matmul check: OK | shape = (64, 64)
Repo TFT modules import: OK!


In [9]:
import os, subprocess

REPO_ROOT = "/content/frn-50k-baseline"
TFT_DIR   = os.path.join(REPO_ROOT, "demand_forecasting", "TFT")
PY310_BIN = "/content/micromamba/envs/tft_py310/bin/python"

# Reset sạch thư mục TFT về nguyên bản trước khi patch
subprocess.run(["git", "-C", REPO_ROOT, "checkout", "--", "demand_forecasting/TFT"], check=True)

train_path   = os.path.join(TFT_DIR, "trainTFT.py")
predict_path = os.path.join(TFT_DIR, "predictTFT.py")

# --- 1. Patch trainTFT.py ---
with open(train_path, "r", encoding="utf-8") as f:
    t_code = f.read()

old_ckpt = '''def get_checkpoint_callback(data_type='censored'):
    return ModelCheckpoint(
        dirpath=f"./lightning_logs/{data_type}/checkpoints",
        filename="{epoch}-{step}",  # Filename format
        save_top_k=-1,  # Save all checkpoints (do not delete old files)
        every_n_epochs=1,  # Save once every epoch (optional, default is 1)
)'''

new_ckpt = '''def get_checkpoint_callback(data_type='censored'):
    log_root = os.environ.get("TFT_LIGHTNING_LOGS_DIR", "./lightning_logs")
    return ModelCheckpoint(
        dirpath=os.path.join(log_root, data_type, "checkpoints"),
        filename="{epoch}-{step}",  # Filename format
        save_top_k=-1,  # Save all checkpoints (do not delete old files)
        every_n_epochs=1,  # Save once every epoch (optional, default is 1)
)'''
assert old_ckpt in t_code, "Không khớp get_checkpoint_callback trong trainTFT.py!"
t_code = t_code.replace(old_ckpt, new_ckpt, 1)

assert "config.num_workers = 32" in t_code
t_code = t_code.replace(
    "config.num_workers = 32",
    'config.num_workers = int(os.environ.get("TFT_NUM_WORKERS", "4"))',
    1,
)

assert 'config.trainer_config["max_epochs"] = 5' in t_code
t_code = t_code.replace(
    'config.trainer_config["max_epochs"] = 5',
    'config.trainer_config["max_epochs"] = int(os.environ.get("TFT_MAX_EPOCHS", "5"))',
    1,
)

assert 'config.trainer_config["default_root_dir"] = root_path' in t_code
t_code = t_code.replace(
    'config.trainer_config["default_root_dir"] = root_path',
    'config.trainer_config["default_root_dir"] = os.environ.get("TFT_LIGHTNING_LOGS_DIR", root_path)',
    1,
)

old_train_load = '''        dataset = load_dataset("Dingdong-Inc/FreshRetailNet-50K")
        df = dataset['train'].to_pandas()'''

new_train_load = '''        local_train = os.environ.get("FRN_LOCAL_TRAIN_PARQUET", "")
        if local_train:
            print(f"[trainTFT] Reading local train parquet: {local_train}", flush=True)
            df = pd.read_parquet(local_train)
        else:
            dataset = load_dataset("Dingdong-Inc/FreshRetailNet-50K")
            df = dataset['train'].to_pandas()'''
assert old_train_load in t_code, "Không khớp đoạn load_dataset trong trainTFT.py!"
t_code = t_code.replace(old_train_load, new_train_load, 1)

old_train_dow = "    df['day_of_week'] = df['dt'].apply(lambda x : datetime.strptime(x, '%Y-%m-%d').weekday())"
new_train_dow = """    smoke_n = int(os.environ.get("TFT_SMOKE_N_PAIRS", "0"))
    if smoke_n > 0:
        smoke_pairs = (
            df[['store_id', 'product_id']].drop_duplicates()
            .sort_values(['store_id', 'product_id'])
            .head(smoke_n)
        )
        df = df.merge(smoke_pairs, on=['store_id', 'product_id'], how='inner')
        print(f"[trainTFT][SMOKE] Filtered to {smoke_n} pairs -> {len(df)} rows", flush=True)
    df['day_of_week'] = pd.to_datetime(df['dt']).dt.weekday"""
assert old_train_dow in t_code, "Không khớp đoạn day_of_week trong trainTFT.py!"
t_code = t_code.replace(old_train_dow, new_train_dow, 1)

with open(train_path, "w", encoding="utf-8") as f:
    f.write(t_code)

# --- 2. Patch predictTFT.py ---
with open(predict_path, "r", encoding="utf-8") as f:
    p_code = f.read()

assert "config.num_workers = 32" in p_code
p_code = p_code.replace(
    "config.num_workers = 32",
    'config.num_workers = int(os.environ.get("TFT_NUM_WORKERS", "4"))',
    1,
)

# QUAN TRỌNG: Nạp train.parquet gốc trước để tính df_train_psd và merge vào df_eval TRƯỚC KHI nạp demand_timesnet.parquet!
old_pred_load = '''    dataset = load_dataset("Dingdong-Inc/FreshRetailNet-50K")
    df_train = dataset['train'].to_pandas()
    df_eval = dataset['eval'].to_pandas()'''

new_pred_load = '''    local_train = os.environ.get("FRN_LOCAL_TRAIN_PARQUET", "")
    local_eval = os.environ.get("FRN_LOCAL_EVAL_PARQUET", "")
    if local_train and local_eval:
        print(f"[predictTFT] Reading local train: {local_train}", flush=True)
        print(f"[predictTFT] Reading local eval : {local_eval}", flush=True)
        df_train = pd.read_parquet(local_train)
        df_eval = pd.read_parquet(local_eval)
    else:
        dataset = load_dataset("Dingdong-Inc/FreshRetailNet-50K")
        df_train = dataset['train'].to_pandas()
        df_eval = dataset['eval'].to_pandas()'''
assert old_pred_load in p_code, "Không khớp đoạn load_dataset trong predictTFT.py!"
p_code = p_code.replace(old_pred_load, new_pred_load, 1)

old_pred_concat = '''    df = pd.concat([df_train, df_eval], ignore_index=True)
    df['day_of_week'] = df['dt'].apply(lambda x : datetime.strptime(x, '%Y-%m-%d').weekday())'''

new_pred_concat = '''    smoke_n = int(os.environ.get("TFT_SMOKE_N_PAIRS", "0"))
    if smoke_n > 0:
        smoke_pairs = (
            df_train[['store_id', 'product_id']].drop_duplicates()
            .sort_values(['store_id', 'product_id'])
            .head(smoke_n)
        )
        df_train = df_train.merge(smoke_pairs, on=['store_id', 'product_id'], how='inner')
        df_eval = df_eval.merge(smoke_pairs, on=['store_id', 'product_id'], how='inner')
        print(f"[predictTFT][SMOKE] Filtered to {smoke_n} pairs -> train={len(df_train)}, eval={len(df_eval)}", flush=True)

    df = pd.concat([df_train, df_eval], ignore_index=True)
    df['day_of_week'] = pd.to_datetime(df['dt']).dt.weekday'''
assert old_pred_concat in p_code, "Không khớp đoạn concat/day_of_week trong predictTFT.py!"
p_code = p_code.replace(old_pred_concat, new_pred_concat, 1)

old_model_dir = '    model_dir = f"./lightning_logs/{data_type}/checkpoints"'
new_model_dir = '''    log_root = os.environ.get("TFT_LIGHTNING_LOGS_DIR", "./lightning_logs")
    model_dir = os.path.join(log_root, data_type, "checkpoints")'''
assert old_model_dir in p_code, "Không khớp model_dir trong predictTFT.py!"
p_code = p_code.replace(old_model_dir, new_model_dir, 1)

old_res_save = '    df_res.to_parquet(f"./{data_type}_res_last_epoch.parquet")'
new_res_save = '''    out_dir = os.environ.get("TFT_OUT_DIR", ".")
    os.makedirs(out_dir, exist_ok=True)
    res_out_path = os.path.join(out_dir, f"{data_type}_res_last_epoch.parquet")
    df_res.to_parquet(res_out_path)
    print(f"[predictTFT] Saved detailed results to: {res_out_path}", flush=True)'''
assert old_res_save in p_code, "Không khớp dòng lưu parquet trong predictTFT.py!"
p_code = p_code.replace(old_res_save, new_res_save, 1)

with open(predict_path, "w", encoding="utf-8") as f:
    f.write(p_code)

# Lưu code_changes.diff lên Drive (bàn giao Git)
diff_text = subprocess.check_output(["git", "-C", REPO_ROOT, "diff", "demand_forecasting/TFT"], text=True)
DIFF_PATH = os.path.join(DRIVE_TASK15_DIR, "code_changes.diff")
with open(DIFF_PATH, "w", encoding="utf-8") as f:
    f.write(diff_text)

print("=== ĐÃ LƯU code_changes.diff TẠI:", DIFF_PATH, "===")
print(diff_text)

=== ĐÃ LƯU code_changes.diff TẠI: /content/drive/MyDrive/PTDLKD/Task15_TFT_Forecasting/code_changes.diff ===
diff --git a/demand_forecasting/TFT/predictTFT.py b/demand_forecasting/TFT/predictTFT.py
index 23e9788..62366a3 100644
--- a/demand_forecasting/TFT/predictTFT.py
+++ b/demand_forecasting/TFT/predictTFT.py
@@ -14,7 +14,7 @@ import argparse
 config.date = '2024-06-26'
 config.quantiles = 7
 config.use_gpu = True
-config.num_workers = 32
+config.num_workers = int(os.environ.get("TFT_NUM_WORKERS", "4"))
 config.dataset_config["max_prediction_length"] = 7
 config.dataset_config["max_encoder_length"] = 70
 target_dates = pd.date_range(start = config.date, periods = config.dataset_config["max_prediction_length"])
@@ -22,9 +22,17 @@ target_dates = [date.strftime("%Y-%m-%d") for date in target_dates]
 
 def loadDataset(data_type='censored', data_path=None):
     t0 = time.time()
-    dataset = load_dataset("Dingdong-Inc/FreshRetailNet-50K")
-    df_train = dataset['train'].to_pandas()
- 

In [17]:
%%bash
set -e
export MPLBACKEND=Agg
PIP_BIN="/content/micromamba/envs/tft_py310/bin/pip"
PY_BIN="/content/micromamba/envs/tft_py310/bin/python"

$PIP_BIN install -q "pandas==1.5.3"
$PY_BIN -c "import pandas, numpy, pyarrow; print('pandas:', pandas.__version__, '| numpy:', numpy.__version__, '| pyarrow:', pyarrow.__version__)"

pandas: 1.5.3 | numpy: 1.26.4 | pyarrow: 25.0.1


In [18]:
import os, shutil, time, subprocess

SMOKE_LOG_PATH = os.path.join(DRIVE_TASK15_DIR, "results", "smoke", "tft_smoke.log")
SMOKE_CKPT_DIR = "/content/tft_smoke_logs"
SMOKE_OUT_DIR  = "/content/tft_smoke_out"
shutil.rmtree(SMOKE_CKPT_DIR, ignore_errors=True)
shutil.rmtree(SMOKE_OUT_DIR, ignore_errors=True)
os.makedirs(SMOKE_CKPT_DIR, exist_ok=True)
os.makedirs(SMOKE_OUT_DIR, exist_ok=True)

def run_tft_pipeline(mode_name, use_demand, log_path, ckpt_root, out_dir, max_epochs=5, smoke_n_pairs=0, num_workers=4):
    env = os.environ.copy()
    env["FRN_LOCAL_TRAIN_PARQUET"] = LOCAL_TRAIN_PARQUET
    env["FRN_LOCAL_EVAL_PARQUET"]  = LOCAL_EVAL_PARQUET
    env["TFT_LIGHTNING_LOGS_DIR"]  = ckpt_root
    env["TFT_OUT_DIR"]             = out_dir
    env["TFT_MAX_EPOCHS"]          = str(max_epochs)
    env["TFT_SMOKE_N_PAIRS"]       = str(smoke_n_pairs)
    env["TFT_NUM_WORKERS"]         = str(num_workers)
    env["PYTHONUNBUFFERED"]        = "1"
    env["MPLBACKEND"]              = "Agg"

    data_type = "recovered" if use_demand else "censored"
    ckpt_sub = os.path.join(ckpt_root, data_type, "checkpoints")
    # Dọn checkpoint cũ của đúng data_type trước khi train mới để start_epoch=-1 không load nhầm
    shutil.rmtree(ckpt_sub, ignore_errors=True)
    os.makedirs(ckpt_sub, exist_ok=True)

    extra_args = ["--demand", "--demand_path", LOCAL_DEMAND_PARQUET] if use_demand else []
    cmds = [
        [PY310_BIN, "-u", "trainTFT.py"] + extra_args,
        [PY310_BIN, "-u", "predictTFT.py"] + extra_args,
    ]

    t_start = time.time()
    os.makedirs(os.path.dirname(log_path), exist_ok=True)
    with open(log_path, "w", encoding="utf-8") as log_f:
        header = (
            f"=== {mode_name} | data_type={data_type} | max_epochs={max_epochs} "
            f"| smoke_n_pairs={smoke_n_pairs} | num_workers={num_workers} ===\n"
        )
        print(header, end="")
        log_f.write(header)
        for cmd in cmds:
            cmd_line = "\n>>> CMD: " + " ".join(cmd) + "\n"
            print(cmd_line, end="")
            log_f.write(cmd_line)
            log_f.flush()
            proc = subprocess.Popen(
                cmd,
                cwd=TFT_DIR,
                env=env,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
                bufsize=1,
            )
            for line in proc.stdout:
                print(line, end="")
                log_f.write(line)
                log_f.flush()
            ret = proc.wait()
            assert ret == 0, f"Lệnh lỗi (exit code {ret}): {' '.join(cmd)}"

    elapsed = time.time() - t_start
    print(f"\n=== HOÀN TẤT {mode_name} trong {elapsed/60:.2f} phút | Log: {log_path} ===")
    return elapsed

# Chạy smoke test 200 cặp (2 epoch)
smoke_elapsed = run_tft_pipeline(
    mode_name="SMOKE_200_PAIRS_RAW",
    use_demand=False,
    log_path=SMOKE_LOG_PATH,
    ckpt_root=SMOKE_CKPT_DIR,
    out_dir=SMOKE_OUT_DIR,
    max_epochs=2,
    smoke_n_pairs=200,
    num_workers=4,
)

=== SMOKE_200_PAIRS_RAW | data_type=censored | max_epochs=2 | smoke_n_pairs=200 | num_workers=4 ===

>>> CMD: /content/micromamba/envs/tft_py310/bin/python -u trainTFT.py
/content/frn-50k-baseline/demand_forecasting/TFT
[trainTFT] Reading local train parquet: /content/frn_cache/train.parquet
[trainTFT][SMOKE] Filtered to 200 pairs -> 18000 rows
18000
load data cost 7.324872255325317s
dataset generation cost 0.30098533630371094
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
model init cost 0.12400102615356445
Missing logger folder: /content/tft_smoke_logs/lightning_logs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

   | Name                               | Type                            | Params
----------------------------------------------------------------------------------------
0  | loss                               | QuantileLoss                    | 0     
1  | logging_metrics 

In [19]:
FULL_CKPT_ROOT = os.path.join(DRIVE_TASK15_DIR, "lightning_logs")
FULL_OUT_DIR   = os.path.join(DRIVE_TASK15_DIR, "results", "full")
RAW_LOG_PATH   = os.path.join(FULL_OUT_DIR, "tft_raw_sales.log")

raw_elapsed = run_tft_pipeline(
    mode_name="FULL_TFT_RawSales",
    use_demand=False,
    log_path=RAW_LOG_PATH,
    ckpt_root=FULL_CKPT_ROOT,
    out_dir=FULL_OUT_DIR,
    max_epochs=5,
    smoke_n_pairs=0,
    num_workers=4,
)

=== FULL_TFT_RawSales | data_type=censored | max_epochs=5 | smoke_n_pairs=0 | num_workers=4 ===

>>> CMD: /content/micromamba/envs/tft_py310/bin/python -u trainTFT.py
/content/frn-50k-baseline/demand_forecasting/TFT
[trainTFT] Reading local train parquet: /content/frn_cache/train.parquet
4500000
load data cost 7.277113199234009s
dataset generation cost 76.55428123474121
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
model init cost 0.2101593017578125
Missing logger folder: /content/drive/MyDrive/PTDLKD/Task15_TFT_Forecasting/lightning_logs/lightning_logs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

   | Name                               | Type                            | Params
----------------------------------------------------------------------------------------
0  | loss                               | QuantileLoss                    | 0     
1  | logging_metrics                

In [20]:
TIMESNET_LOG_PATH = os.path.join(FULL_OUT_DIR, "tft_timesnet.log")

timesnet_elapsed = run_tft_pipeline(
    mode_name="FULL_TFT_TimesNet",
    use_demand=True,
    log_path=TIMESNET_LOG_PATH,
    ckpt_root=FULL_CKPT_ROOT,
    out_dir=FULL_OUT_DIR,
    max_epochs=5,
    smoke_n_pairs=0,
    num_workers=4,
)

=== FULL_TFT_TimesNet | data_type=recovered | max_epochs=5 | smoke_n_pairs=0 | num_workers=4 ===

>>> CMD: /content/micromamba/envs/tft_py310/bin/python -u trainTFT.py --demand --demand_path /content/frn_cache/demand_timesnet.parquet
/content/frn-50k-baseline/demand_forecasting/TFT
4500000
load data cost 8.224964380264282s
dataset generation cost 77.17799019813538
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
model init cost 0.2159574031829834
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

   | Name                               | Type                            | Params
----------------------------------------------------------------------------------------
0  | loss                               | QuantileLoss                    | 0     
1  | logging_metrics                    | ModuleList                      | 0     
2  | input_embeddings                   | MultiEmbedding         

In [21]:
import os, numpy as np, pandas as pd

METRICS_CSV_PATH = os.path.join(DRIVE_TASK15_DIR, "results", "tft_forecasting_metrics.csv")

# Số tham chiếu Bảng 3 bài báo (cột TFT)
PAPER_REF = {
    ("TFT_RawSales", "Overall"):   (31.75, -7.37),
    ("TFT_RawSales", "Low Sale"):  (37.04, -1.72),
    ("TFT_RawSales", "High Sale"): (27.95, -11.46),
    ("TFT_TimesNet", "Overall"):   (29.02,  2.58),
    ("TFT_TimesNet", "Low Sale"):  (37.33,  7.78),
    ("TFT_TimesNet", "High Sale"): (23.03,  0.87),
}

GROUP_MAP = [
    ("psd>=0", "Overall",   206562),
    ("psd<1",  "Low Sale",  153208),
    ("psd>=1", "High Sale",  53354),
]

rows = []
eval_counts_summary = {}

for config_id, data_type in [("TFT_RawSales", "censored"), ("TFT_TimesNet", "recovered")]:
    res_pq = os.path.join(FULL_OUT_DIR, f"{data_type}_res_last_epoch.parquet")
    assert os.path.isfile(res_pq), f"Không tìm thấy {res_pq}!"
    df_res = pd.read_parquet(res_pq)

    # Kiểm tra đủ 7 ngày eval cho mỗi nhóm & tổng dòng hợp lệ khớp 206.562 (153.208 + 53.354)
    eval_counts_summary[config_id] = {}
    for g_key, g_label, expected_cnt in GROUP_MAP:
        sub = df_res[df_res["group_type"] == g_key]
        assert len(sub) == 7, f"{config_id}/{g_label}: kỳ vọng 7 ngày, thực tế {len(sub)}"
        total_cnt = int(sub["sample_cnt"].sum())
        eval_counts_summary[config_id][g_label] = total_cnt
        print(f"[{config_id:12s} | {g_label:9s}] sample_cnt (7 ngày) = {total_cnt:,} (kỳ vọng {expected_cnt:,})")
        assert total_cnt == expected_cnt, f"Sai số dòng eval hợp lệ cho {config_id}/{g_label}: {total_cnt} != {expected_cnt}"

        # Trung bình 7 ngày khớp chính xác bảng in từ predictTFT.py
        wape_pct = round(float(sub["wape"].mean()) * 100.0, 2)
        wpe_pct  = round(float(sub["wpe"].mean()) * 100.0, 2)
        mae_val  = round(float(sub["mae"].mean()), 4)

        assert np.isfinite(wape_pct) and np.isfinite(wpe_pct) and np.isfinite(mae_val), "Chỉ số không hữu hạn!"
        paper_wape, paper_wpe = PAPER_REF[(config_id, g_label)]
        rows.append({
            "config_id": config_id,
            "group": g_label,
            "wape_pct": wape_pct,
            "wpe_pct": wpe_pct,
            "mae": mae_val,
            "paper_wape_pct": paper_wape,
            "paper_wpe_pct": paper_wpe,
            "delta_wape_pp": round(wape_pct - paper_wape, 2),
            "delta_wpe_pp": round(wpe_pct - paper_wpe, 2),
            "status": "completed",
        })

metrics_df = pd.DataFrame(rows)
assert len(metrics_df) == 6 and list(metrics_df.columns) == [
    "config_id", "group", "wape_pct", "wpe_pct", "mae",
    "paper_wape_pct", "paper_wpe_pct", "delta_wape_pp", "delta_wpe_pp", "status"
]
metrics_df.to_csv(METRICS_CSV_PATH, index=False)
print("\nĐã lưu bảng chuẩn 10 cột tại:", METRICS_CSV_PATH)
display(metrics_df)

[TFT_RawSales | Overall  ] sample_cnt (7 ngày) = 206,562 (kỳ vọng 206,562)
[TFT_RawSales | Low Sale ] sample_cnt (7 ngày) = 153,208 (kỳ vọng 153,208)
[TFT_RawSales | High Sale] sample_cnt (7 ngày) = 53,354 (kỳ vọng 53,354)
[TFT_TimesNet | Overall  ] sample_cnt (7 ngày) = 206,562 (kỳ vọng 206,562)
[TFT_TimesNet | Low Sale ] sample_cnt (7 ngày) = 153,208 (kỳ vọng 153,208)
[TFT_TimesNet | High Sale] sample_cnt (7 ngày) = 53,354 (kỳ vọng 53,354)

Đã lưu bảng chuẩn 10 cột tại: /content/drive/MyDrive/PTDLKD/Task15_TFT_Forecasting/results/tft_forecasting_metrics.csv


,config_id,group,wape_pct,wpe_pct,mae,paper_wape_pct,paper_wpe_pct,delta_wape_pp,delta_wpe_pp,status
0,TFT_RawSales,Overall,31.96,-9.85,0.3990,31.75,-7.37,0.21,-2.48,completed
1,TFT_RawSales,Low Sale,36.89,-5.59,0.2588,37.04,-1.72,-0.15,-3.87,completed
2,TFT_RawSales,High Sale,28.41,-12.90,0.8120,27.95,-11.46,0.46,-1.44,completed
3,TFT_TimesNet,Overall,28.93,0.78,0.3595,29.02,2.58,-0.09,-1.80,completed
4,TFT_TimesNet,Low Sale,37.06,6.84,0.2599,37.33,7.78,-0.27,-0.94,completed
5,TFT_TimesNet,High Sale,23.07,-3.57,0.6506,23.03,0.87,0.04,-4.44,completed


In [23]:
import os, json, glob, subprocess, torch
from datetime import datetime, timezone

# ====== DÁN LINK CHIA SẺ GOOGLE DRIVE THƯ MỤC CHECKPOINT lightning_logs/ VÀO ĐÂY ======
CKPT_DRIVE_LINK = "https://drive.google.com/drive/folders/1gPexk9JqLJFH-MUPFE-7d69g9pQa7xD2"

env_cmd = (
    "import sys, json, torch, pytorch_lightning, pytorch_forecasting, numpy, pandas, pyarrow; "
    "print(json.dumps({"
    "'python': sys.version.split()[0], "
    "'torch': torch.__version__, "
    "'pytorch_lightning': pytorch_lightning.__version__, "
    "'pytorch_forecasting': pytorch_forecasting.__version__, "
    "'numpy': numpy.__version__, "
    "'pandas': pandas.__version__, "
    "'pyarrow': pyarrow.__version__"
    "}))"
)
env_info = subprocess.check_output(
    [PY310_BIN, "-c", env_cmd],
    env={**os.environ, "MPLBACKEND": "Agg"},
    text=True,
).strip()
env_versions = json.loads(env_info)

ckpt_raw_files = sorted(glob.glob(os.path.join(FULL_CKPT_ROOT, "censored", "checkpoints", "*.ckpt")))
ckpt_tn_files  = sorted(glob.glob(os.path.join(FULL_CKPT_ROOT, "recovered", "checkpoints", "*.ckpt")))
print("Environment          :", env_versions)
print("Checkpoints censored :", [os.path.basename(x) for x in ckpt_raw_files])
print("Checkpoints recovered:", [os.path.basename(x) for x in ckpt_tn_files])

now_utc = datetime.now(timezone.utc).isoformat()
gpu_name = torch.cuda.get_device_name(0)

# 1. Lưu results/run_metadata.json
metadata = {
    "task": "Task 15 - TFT Forecasting (TFT_RawSales & TFT_TimesNet)",
    "author": "Minh Nguyễn",
    "branch": "feature/task15-tft-forecasting",
    "timestamp_utc": now_utc,
    "baseline_commit": "8af694a09a5a427d1d50648d348c022565a781f2",
    "hardware": {"gpu": gpu_name, "colab_high_ram": True},
    "environment": env_versions,
    "hyperparameters": {
        "seed": 2025,
        "date": "2024-06-26",
        "max_encoder_length": 70,
        "max_prediction_length": 7,
        "batch_size": 1024,
        "hidden_size": 128,
        "attention_head_size": 8,
        "lstm_layers": 2,
        "dropout": 0.1,
        "learning_rate": 0.01,
        "optimizer": "adamw",
        "weight_decay": 1e-4,
        "quantiles_ratio": [0.1, 0.3, 0.5, 0.7, 0.9],
        "max_epochs": 5,
        "num_workers": 4,
    },
    "inputs_sha256": EXPECTED_SHA256,
    "eval_valid_sample_counts": eval_counts_summary,
    "runtime_minutes": {
        "smoke_200_pairs": round(smoke_elapsed / 60.0, 2),
        "full_tft_raw_sales": round(raw_elapsed / 60.0, 2),
        "full_tft_timesnet": round(timesnet_elapsed / 60.0, 2),
    },
    "checkpoints_saved": {
        "censored": [os.path.basename(x) for x in ckpt_raw_files],
        "recovered": [os.path.basename(x) for x in ckpt_tn_files],
        "drive_link": CKPT_DRIVE_LINK,
    },
    "metrics": metrics_df.to_dict(orient="records"),
}

META_JSON_PATH = os.path.join(DRIVE_TASK15_DIR, "results", "run_metadata.json")
with open(META_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)
print("Đã tạo:", META_JSON_PATH)

# 2. Tạo run_info.md
FENCE = "`" * 3
run_info_lines = [
    "# Task 15 — Run Info (TFT Forecasting: `TFT_RawSales` & `TFT_TimesNet`)",
    "",
    f"- **Thời điểm hoàn tất (UTC):** {now_utc}",
    "- **Người thực hiện:** Minh Nguyễn | **Nhánh:** `feature/task15-tft-forecasting`",
    f"- **Phần cứng:** Google Colab GPU `{gpu_name}` (High-RAM)",
    f"- **Môi trường Python 3.10 độc lập (`micromamba`):** Python `{env_versions['python']}`, PyTorch `{env_versions['torch']}`, PyTorch Lightning `{env_versions['pytorch_lightning']}`, PyTorch Forecasting `{env_versions['pytorch_forecasting']}`, NumPy `{env_versions['numpy']}`, Pandas `{env_versions['pandas']}`, PyArrow `{env_versions['pyarrow']}`",
    "- **Mã tác giả đối chiếu (`frn-50k-baseline`):** commit `8af694a09a5a427d1d50648d348c022565a781f2`",
    "- **Bản vá áp dụng:** Xem `code_changes.diff` (nạp Parquet local giữ đúng thứ tự tính `df_train_psd` trước khi thay `sale_amount_pred`, vectorize `day_of_week`, `num_workers=4`, tách thư mục `lightning_logs` lên Drive và hỗ trợ smoke 200 cặp).",
    "",
    "## Đầu vào đã kiểm định SHA-256",
    f"- `train.parquet`: `{EXPECTED_SHA256['train.parquet']}`",
    f"- `eval.parquet`: `{EXPECTED_SHA256['eval.parquet']}`",
    f"- `demand_timesnet.parquet`: `{EXPECTED_SHA256['demand_timesnet.parquet']}`",
    "",
    "## Lệnh chạy chính thức",
    f"{FENCE}bash",
    "# Cấu hình 1: TFT_RawSales (censored)",
    "python -u trainTFT.py",
    "python -u predictTFT.py",
    "",
    "# Cấu hình 2: TFT_TimesNet (recovered)",
    "python -u trainTFT.py --demand --demand_path /content/frn_cache/demand_timesnet.parquet",
    "python -u predictTFT.py --demand --demand_path /content/frn_cache/demand_timesnet.parquet",
    FENCE,
    "",
    "## Thời gian chạy & kiểm định",
    f"- Smoke 200 cặp (2 epoch): `{smoke_elapsed/60:.2f}` phút — log: `results/smoke/tft_smoke.log`",
    f"- Full `TFT_RawSales` (5 epoch + predict): `{raw_elapsed/60:.2f}` phút — log: `results/full/tft_raw_sales.log`",
    f"- Full `TFT_TimesNet` (5 epoch + predict): `{timesnet_elapsed/60:.2f}` phút — log: `results/full/tft_timesnet.log`",
    "- Số dòng eval hợp lệ (`stock_hour6_22_cnt == 0`): **206.562** (`153.208` Low Sale + `53.354` High Sale) ở cả 2 cấu hình.",
    "",
    "## Liên kết checkpoint trên Google Drive (không commit lên Git)",
    f"- Thư mục `lightning_logs/` (gồm `censored/checkpoints/*.ckpt` và `recovered/checkpoints/*.ckpt`): {CKPT_DRIVE_LINK}",
]
RUN_INFO_PATH = os.path.join(DRIVE_TASK15_DIR, "run_info.md")
with open(RUN_INFO_PATH, "w", encoding="utf-8") as f:
    f.write("\n".join(run_info_lines) + "\n")
print("Đã tạo:", RUN_INFO_PATH)

# 3. Tạo Task15_TFT_Forecasting/README.md
readme_lines = [
    "# Task 15 — Dự báo SOTA bằng Temporal Fusion Transformer (TFT)",
    "",
    "Thư mục chứa notebook, bản vá, log và kết quả tái lập cột **TFT** (Bảng 3 bài báo FreshRetailNet-50K) cho 2 cấu hình trọng tâm:",
    "- **`TFT_RawSales`** (`data_type='censored'`): huấn luyện trực tiếp trên doanh số bị che khuất `train.parquet`.",
    "- **`TFT_TimesNet`** (`data_type='recovered'`): huấn luyện trên nhu cầu đã khôi phục `sale_amount_pred` từ `demand_timesnet.parquet` (Task 10).",
    "",
    "## Cấu trúc thư mục",
    f"{FENCE}text",
    "Task15_TFT_Forecasting/",
    "├── README.md",
    "├── run_info.md",
    "├── code_changes.diff",
    "├── notebooks/",
    "│   └── 15_tft_forecasting.ipynb",
    "└── results/",
    "    ├── tft_forecasting_metrics.csv",
    "    ├── run_metadata.json",
    "    ├── smoke/tft_smoke.log",
    "    └── full/",
    "        ├── tft_raw_sales.log",
    "        └── tft_timesnet.log",
    FENCE,
    "",
    "## Cách chạy lại",
    "1. Mở `notebooks/15_tft_forecasting.ipynb` trên Google Colab với runtime **NVIDIA L4 GPU (High-RAM)**.",
    "2. Cập nhật 4 đường dẫn Google Drive ở Cell 2 (khớp SHA-256 của `train.parquet`, `eval.parquet`, `demand_timesnet.parquet`).",
    "3. Chạy tuần tự Cell 1 → Cell 10 (môi trường Python 3.10 + `pytorch-lightning==1.9.4` + `pandas==1.5.3` được khởi tạo tự động bằng `micromamba`).",
]
README_PATH = os.path.join(DRIVE_TASK15_DIR, "README.md")
with open(README_PATH, "w", encoding="utf-8") as f:
    f.write("\n".join(readme_lines) + "\n")
print("Đã tạo:", README_PATH)

# 4. Tạo reports/Task15_Report.md
def get_row(cid, grp):
    return metrics_df[(metrics_df["config_id"] == cid) & (metrics_df["group"] == grp)].iloc[0]

def fmt_sign(v):
    return f"+{v:.2f}" if v > 0 else f"{v:.2f}"

table_rows = []
for grp in ["Overall", "Low Sale", "High Sale"]:
    for cid in ["TFT_RawSales", "TFT_TimesNet"]:
        r = get_row(cid, grp)
        table_rows.append(
            f"| **{grp}** | `{cid}` | {r['wape_pct']:.2f}% | {fmt_sign(r['wpe_pct'])}% | {r['mae']:.4f} "
            f"| {r['paper_wape_pct']:.2f}% | {fmt_sign(r['paper_wpe_pct'])}% "
            f"| {fmt_sign(r['delta_wape_pp'])} pp | {fmt_sign(r['delta_wpe_pp'])} pp |"
        )

r_ov_raw = get_row("TFT_RawSales", "Overall")
r_ov_tn  = get_row("TFT_TimesNet", "Overall")
r_hi_raw = get_row("TFT_RawSales", "High Sale")
r_hi_tn  = get_row("TFT_TimesNet", "High Sale")
r_lo_raw = get_row("TFT_RawSales", "Low Sale")
r_lo_tn  = get_row("TFT_TimesNet", "Low Sale")

report_lines = [
    "# Báo cáo Task 15: Dự báo SOTA bằng Temporal Fusion Transformer (`TFT_RawSales` & `TFT_TimesNet`)",
    "",
    "**Người thực hiện:** Minh Nguyễn | **Nhánh:** `feature/task15-tft-forecasting` | **Thư mục:** `Task15_TFT_Forecasting/`",
    "",
    "## 1. Thiết lập thực nghiệm",
    f"- **Môi trường & phần cứng:** Google Colab GPU `{gpu_name}` (High-RAM), môi trường Python `{env_versions['python']}` độc lập qua `micromamba` với `torch=={env_versions['torch']}`, `pytorch-lightning=={env_versions['pytorch_lightning']}`, `pytorch-forecasting=={env_versions['pytorch_forecasting']}`, `pandas=={env_versions['pandas']}` để bảo toàn 100% mã huấn luyện gốc (`pl.Trainer(..., gpus=1)` và hook Lightning 1.x).",
    "- **Dữ liệu đầu vào chuẩn:** `train.parquet` (`b7e01c71...`), `eval.parquet` (`61cfd3cb...`) và `demand_timesnet.parquet` (`c34f4eafe087...` từ [Task 10](Task10_Report.md)).",
    "- **Cấu hình mô hình (`configs/tft_config.py` + `trainTFT.py`):** `seed=2025`, `max_encoder_length=70`, `max_prediction_length=7`, `batch_size=1024`, `hidden_size=128`, `attention_head_size=8`, `lstm_layers=2`, `dropout=0.1`, `lr=0.01`, `optimizer=adamw`, `weight_decay=1e-4`, `max_epochs=5`, `num_workers=4`.",
    "- **Quy trình đánh giá:** Cả hai lượt hoàn tất 5 epoch huấn luyện; `predictTFT.py` tính `psd` từ `train.parquet` gốc trước khi nạp `demand_timesnet.parquet` để đồng bộ tuyệt đối phân nhóm `Low Sale` (`psd < 1`, **153.208** dòng), `High Sale` (`psd >= 1`, **53.354** dòng) và `Overall` (**206.562** dòng `stock_hour6_22_cnt == 0`).",
    "",
    "## 2. Kết quả định lượng và Đối chiếu Bảng 3",
    "| Phân nhóm | Cấu hình | WAPE thực chạy | WPE thực chạy | MAE | WAPE bài báo | WPE bài báo | Chênh lệch WAPE | Chênh lệch WPE |",
    "| :--- | :--- | :---: | :---: | :---: | :---: | :---: | :---: | :---: |",
    *table_rows,
    "",
    "## 3. Phân tích so sánh `TFT_RawSales` và `TFT_TimesNet`",
    f"1. **Cải thiện toàn cục (Overall):** Khi huấn luyện TFT trên nhu cầu đã khôi phục từ TimesNet thay cho doanh số bị che khuất, WAPE giảm từ **{r_ov_raw['wape_pct']:.2f}%** xuống **{r_ov_tn['wape_pct']:.2f}%** (**{r_ov_tn['wape_pct'] - r_ov_raw['wape_pct']:+.2f} điểm phần trăm**) và MAE thay đổi từ **{r_ov_raw['mae']:.4f}** xuống **{r_ov_tn['mae']:.4f}**; đồng thời WPE chuyển từ dự báo thiếu sâu (**{fmt_sign(r_ov_raw['wpe_pct'])}%**) lên **{fmt_sign(r_ov_tn['wpe_pct'])}%**.",
    f"2. **Đột phá ở nhóm bán chạy (High Sale):** Nhóm mặt hàng chủ lực có tỉ lệ hết hàng cao ghi nhận mức cải thiện mạnh nhất — WAPE giảm từ **{r_hi_raw['wape_pct']:.2f}%** xuống **{r_hi_tn['wape_pct']:.2f}%** (**{r_hi_tn['wape_pct'] - r_hi_raw['wape_pct']:+.2f} điểm phần trăm**), MAE giảm từ **{r_hi_raw['mae']:.4f}** xuống **{r_hi_tn['mae']:.4f}**, và thiên lệch dự báo thiếu giảm mạnh từ **{fmt_sign(r_hi_raw['wpe_pct'])}%** về sát 0 (**{fmt_sign(r_hi_tn['wpe_pct'])}%**).",
    f"3. **Đánh đổi ở nhóm bán chậm (Low Sale):** Do dữ liệu bán chậm thưa thớt theo giờ, `TFT_TimesNet` đạt WAPE **{r_lo_tn['wape_pct']:.2f}%** so với **{r_lo_raw['wape_pct']:.2f}%** của `TFT_RawSales`, và WPE chuyển dương (**{fmt_sign(r_lo_tn['wpe_pct'])}%** so với **{fmt_sign(r_lo_raw['wpe_pct'])}%**), nhất quán với xu hướng công bố trong bài báo gốc.",
    "4. **Mức độ tái lập so với Bảng 3:** Cả hai cấu hình tái lập sát kết quả SOTA công bố của bài báo; sai lệch nhỏ (nếu có) xuất phát từ tính không tất định của GPU/cuDNN trên phần cứng L4 khác môi trường gốc và sai lệch rất nhỏ ở đầu vào `demand_timesnet.parquet` từ Task 10.",
    "",
    "## 4. Bàn giao",
    "- [Notebook đã chạy](../Task15_TFT_Forecasting/notebooks/15_tft_forecasting.ipynb), [bản vá mã nguồn](../Task15_TFT_Forecasting/code_changes.diff), [bảng chỉ số chuẩn 10 cột](../Task15_TFT_Forecasting/results/tft_forecasting_metrics.csv).",
    "- [Log smoke 200 cặp](../Task15_TFT_Forecasting/results/smoke/tft_smoke.log), [log full `TFT_RawSales`](../Task15_TFT_Forecasting/results/full/tft_raw_sales.log), [log full `TFT_TimesNet`](../Task15_TFT_Forecasting/results/full/tft_timesnet.log), [metadata](../Task15_TFT_Forecasting/results/run_metadata.json), [biên bản chạy](../Task15_TFT_Forecasting/run_info.md) và [README](../Task15_TFT_Forecasting/README.md).",
    f"- Checkpoint `.ckpt` trong `lightning_logs/` lưu trên Google Drive (không commit lên Git): {CKPT_DRIVE_LINK}.",
]
REPORT_PATH = os.path.join(DRIVE_TASK15_DIR, "reports", "Task15_Report.md")
with open(REPORT_PATH, "w", encoding="utf-8") as f:
    f.write("\n".join(report_lines) + "\n")
print("Đã tạo:", REPORT_PATH)
print("\n=== XEM TRƯỚC BẢNG KẾT QUẢ ===")
display(metrics_df)

Environment          : {'python': '3.10.22', 'torch': '2.0.1+cu118', 'pytorch_lightning': '1.9.4', 'pytorch_forecasting': '1.0.0', 'numpy': '1.26.4', 'pandas': '1.5.3', 'pyarrow': '25.0.1'}
Checkpoints censored : ['epoch=0-step=683.ckpt', 'epoch=1-step=1366.ckpt', 'epoch=2-step=2049.ckpt', 'epoch=3-step=2732.ckpt', 'epoch=4-step=3415.ckpt']
Checkpoints recovered: ['epoch=0-step=683.ckpt', 'epoch=1-step=1366.ckpt', 'epoch=2-step=2049.ckpt', 'epoch=3-step=2732.ckpt', 'epoch=4-step=3415.ckpt']
Đã tạo: /content/drive/MyDrive/PTDLKD/Task15_TFT_Forecasting/results/run_metadata.json
Đã tạo: /content/drive/MyDrive/PTDLKD/Task15_TFT_Forecasting/run_info.md
Đã tạo: /content/drive/MyDrive/PTDLKD/Task15_TFT_Forecasting/README.md
Đã tạo: /content/drive/MyDrive/PTDLKD/Task15_TFT_Forecasting/reports/Task15_Report.md

=== XEM TRƯỚC BẢNG KẾT QUẢ ===


,config_id,group,wape_pct,wpe_pct,mae,paper_wape_pct,paper_wpe_pct,delta_wape_pp,delta_wpe_pp,status
0,TFT_RawSales,Overall,31.96,-9.85,0.3990,31.75,-7.37,0.21,-2.48,completed
1,TFT_RawSales,Low Sale,36.89,-5.59,0.2588,37.04,-1.72,-0.15,-3.87,completed
2,TFT_RawSales,High Sale,28.41,-12.90,0.8120,27.95,-11.46,0.46,-1.44,completed
3,TFT_TimesNet,Overall,28.93,0.78,0.3595,29.02,2.58,-0.09,-1.80,completed
4,TFT_TimesNet,Low Sale,37.06,6.84,0.2599,37.33,7.78,-0.27,-0.94,completed
5,TFT_TimesNet,High Sale,23.07,-3.57,0.6506,23.03,0.87,0.04,-4.44,completed
